## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks access and extracts data from the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

from datetime import date
from tqdm import tqdm

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, get_catalog

In [ ]:
# Selected features
selected_features = [
    'pet'
]

# Save path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/processed/'

# MS Planetary Catalog Collection
catalog = get_catalog(
    'https://planetarycomputer.microsoft.com/api/stac/v1'
)

In [5]:
# Dataframe for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

# Dataframe for testing
validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
)

### Functions

In [2]:
def load_terraclimate_dataset():
    '''
    Opens the TerraClimate Zarr/NetCDF dataset from the
    Microsoft Planetary Computer, handling storage options
    automatically.
    '''
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
def filterg(ds, var):
    '''
    Filters the dataset for the desired time range (2011–2015) and the
    spatial extent corresponding to the study region. The resulting data
    is converted into a pandas DataFrame with standardized column names.
    '''
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final

In [4]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

In [ ]:
def process_df(destination_df, original_df):
    '''
    Just 'unions' the databases
    '''
    # Finalizing df
    destination_df['Latitude'] = original_df['Latitude']
    destination_df['Longitude'] = original_df['Longitude']
    destination_df['Sample Date'] = original_df['Sample Date']
    cols_to_keep = [
        'Latitude', 'Longitude', 'Sample Date'
    ] + selected_features
    
    return destination_df[selected_features]

## Data transformation

In [7]:
# Load TerraClimate dataset, filter (time,region,parameter),
# filter for nearest parameter values
ds = load_terraclimate_dataset()
tc_parameter = filterg(ds, 'pet')

In [ ]:
terraclimate_training_df = assign_nearest_climate(water_quality_df, tc_parameter, 'pet')
terraclimate_validation_df = assign_nearest_climate(validation_df, tc_parameter, 'pet')

In [8]:
terraclimate_training_df_processed = process_df(terraclimate_training_df, water_quality_df)
terraclimate_validation_df_processed = process_df(terraclimate_validation_df, validation_df)

## Data saving

In [ ]:
save_df(terraclimate_training_df, 'terraclimate_train_features', save_path)
save_df(terraclimate_validation_df, 'terraclimate_val_features', save_path)